# Pelican file events: live EarthScope GNSS data

This notebook subscribes to a Pelican namespace and acts on each new
object as it appears. The data is EarthScope GNSS displacement: east,
north and up, one file per minute.

`ndp-ep` gives you three things: it tells you when an object appears,
and it reads that object into memory or fetches it to disk. The bytes
travel from the federation straight to this process — the NDP Endpoint
is not in the data path, and there is no STOMP, no WebSocket handling
and no `asyncio` to write.

What you do with the object afterwards is your own workflow. The
second half of this notebook plots it, but that part is an example,
not part of the library.

In [ ]:
# Not on PyPI yet: the Pelican support lives on this branch.
%pip install "ndp-ep[pelican] @ git+https://github.com/sci-ndp/ndp-ep-py.git@feature/pelican-direct-data-access"

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

## Configuration

`CLIENT_ID` identifies your subscriber, and the event server keeps one
queue per id. Sharing an id with another reader is worse than it
sounds: you compete for the same events, and you inherit whatever that
queue has already accumulated. The default below derives an id from
your local username so that does not happen. Keep it stable between
runs — a new id every time leaves an abandoned queue behind on the
server.

`EVENT_SOURCE` is the namespace to watch. It is a rolling window: it
holds the last 100 objects, roughly the last hour and forty minutes,
and the publisher adds one per minute while the oldest is deleted. An
event can therefore point at an object that no longer exists, which is
why the loop further down tolerates a failed read.

The credentials are checked by the event server against its own store;
they are unrelated to the Endpoint token, and will be replaced by an
access token later.

In [ ]:
import getpass

# One queue per id, so this must not be shared with other readers.
CLIENT_ID = f"{getpass.getuser()}-pelican-demo"
EVENT_SOURCE = "osdf/vdc/public/pelican_protocol"

ENDPOINT_URL = "http://155.101.6.191:8003"
EVENT_USERNAME = "your-username"
EVENT_PASSWORD = "your-password"

## Connect and subscribe

One client object covers everything that follows: the subscription
that tells you an object appeared, and the reads that fetch it.

In [ ]:
from ndp_ep import APIClient

client = APIClient(base_url=ENDPOINT_URL)

subscription = client.subscribe_pelican(
    EVENT_SOURCE,
    client_id=CLIENT_ID,
    username=EVENT_USERNAME,
    password=EVENT_PASSWORD,
)

subscription.wait_until_connected(timeout=30)
subscription.status

## What an event carries

Each event names the object, gives a reference ready to pass to
`pelican_read` or `pelican_fetch`, and reports its size and
modification time.

`subscription.events(timeout=...)` blocks until the next event and
yields it once. Redeliveries are suppressed against a record on disk,
so restarting the notebook will not reprocess what it already handled.

Connecting is fast, but the first event is not: the event server
usually delivers nothing for the first few minutes and then sends what
it has queued in a burst. Measured runs waited between two and four
minutes. The `timeout` bounds the wait so the cell always ends instead
of hanging, and it has to be comfortably larger than that delay.

In [ ]:
EVENT_TIMEOUT = 420  # seconds; well above the startup delay

for received, event in enumerate(
    subscription.events(timeout=EVENT_TIMEOUT), 1
):
    print(event.name, event.size, event.mod_time)
    print("   ", event.url)
    latest = event.url
    if received >= 2:
        break

## Read the object, without touching disk

`pelican_read` returns the bytes. Nothing is written anywhere, and the
Endpoint never sees the payload.

References are accepted in any of the spellings that turn up in
practice: a bare path, `osdf://...`, or `pelican://host/...`. The
`url` an event hands you is already one of them.

In [ ]:
raw = client.pelican_read(latest)

print(len(raw), "bytes")
print(raw[:200].decode())

## Or fetch it to disk

`pelican_fetch` is the same operation with a file as the destination.
An existing target is an error rather than a silent overwrite.

In [ ]:
path = client.pelican_fetch(latest, "./data/")

print(path, path.stat().st_size, "bytes")

## Listing, without subscribing

The namespace can also be listed directly, with no subscription
involved.

Note that `pelican_list` returns names in lexicographic order, not
chronological order: `AGMT.CI.LY_.20_c100.csv` sorts before
`AGMT.CI.LY_.20_c99.csv`. The end of the list is not the newest data —
use an event's `url` when that is what you want.

In [ ]:
objects = client.pelican_list(EVENT_SOURCE)

print(f"{len(objects)} objects")
objects[:3]

---

# ⬆ Everything above is `ndp-ep` — ⬇ everything below is your own code

The library's job ends here: you know an object appeared, and you can
read it or fetch it. What follows is an ordinary user workflow built
on top of that — nothing below is part of the Endpoint or the client
library.

---

## Example: plot each file as it arrives

The same loop as above, except that each object is parsed and drawn
instead of printed. Raise `MAX_EVENTS` to keep it running longer.

The namespace keeps only its last 100 objects, so an event can name
one that has already been deleted. The loop reports and skips those
rather than stopping.

In [ ]:
%pip install plotly anywidget pandas

In [ ]:
import io

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

MAX_POINTS = 600
MAX_EVENTS = 3  # EVENT_TIMEOUT comes from the cell above

all_data = pd.DataFrame()


def create_figure(title, color):
    fig = go.FigureWidget()
    fig.add_scatter(mode="lines", line=dict(color=color, width=2),
                    name=title)
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=250,
        margin=dict(l=50, r=30, t=40, b=40),
        xaxis_title="Time",
        yaxis_title=title,
    )
    return fig


east_fig = create_figure("East", "royalblue")
north_fig = create_figure("North", "green")
up_fig = create_figure("Up", "firebrick")

display(east_fig)
display(north_fig)
display(up_fig)

received = 0

for event in subscription.events(timeout=EVENT_TIMEOUT):

    try:
        raw = client.pelican_read(event.url)
    except ValueError as exc:
        # The namespace keeps only the last 100 objects. An event can
        # arrive for one that has already been rotated out.
        print(f"Skipped {event.name}: {exc}")
        continue

    print(f"Received {event.name}")

    df = pd.read_csv(io.BytesIO(raw))
    df["datetime"] = pd.to_datetime(df["time"], unit="ms", utc=True)

    all_data = pd.concat([all_data, df], ignore_index=True)
    if len(all_data) > MAX_POINTS:
        all_data = all_data.iloc[-MAX_POINTS:].copy()

    x = all_data["datetime"]
    for figure, column in (
        (east_fig, "east"),
        (north_fig, "north"),
        (up_fig, "up"),
    ):
        with figure.batch_update():
            figure.data[0].x = x
            figure.data[0].y = all_data[column]

    received += 1
    if received >= MAX_EVENTS:
        break

## All three components on one figure

In [ ]:
combined = go.FigureWidget()

for column, color in (("east", "royalblue"),
                     ("north", "green"),
                     ("up", "firebrick")):
    combined.add_scatter(
        x=all_data["datetime"],
        y=all_data[column],
        name=column.capitalize(),
        line=dict(color=color, width=2),
    )

combined.update_layout(template="plotly_white", height=350,
                       xaxis_title="Time")
display(combined)

## Closing

Closing performs the WebSocket closing handshake, so the event server
does not sit on a half-open connection. Using the subscription as a
context manager (`with client.subscribe_pelican(...) as subscription:`)
does this automatically.

In [ ]:
subscription.close()
subscription.status["metrics"]